<a href="https://colab.research.google.com/github/ivansantoscoy/analisisdatarotacion/blob/main/Analisis_Rotacion_Personal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏭 Análisis de Rotación de Personal
## Pipeline completo: PCA → Estadístico → Pareto 80/20 → Causa Raíz

---
---

| Sección | Qué hace |
|---|---|
| 0 | Instala librerías |
| 1 | Carga y configura el archivo |
| 2 | Limpieza y procesamiento |
| 3 | PCA — variables que más pesan |
| 4 | Análisis estadístico por grupos |
| 5 | Pareto 80/20 (3 dimensiones) |
| 6 | Causa raíz — 5 Porqués con datos |
| 7 | Exportar Excel de resultados |


## Sección 0 — Instalación de librerías
> Ejecuta esta celda solo la primera vez

In [ ]:
!pip install openpyxl scikit-learn pandas numpy matplotlib seaborn -q
print("✅ Librerías instaladas correctamente")


## Sección 1 — Carga del archivo y configuración
> **Aquí es donde pegas tu configuración.** El resto del notebook se adapta automáticamente.


In [ ]:
# ══════════════════════════════════════════════
# CELDA DE CONFIGURACIÓN — Solo modificar aquí
# ══════════════════════════════════════════════

# Nombres de columnas en tu archivo (ajusta si cambian)
CONFIG = {
    # Columna de antigüedad en semanas
    'col_antiguedad':      'Antigüedad en Semanas',
    # Columna de total de faltas
    'col_faltas':          'Total de faltas',
    # Columna de permisos
    'col_permisos':        'Permisos',
    # Columna de salario
    'col_salario':         'Salario',
    # Columna de turno
    'col_turno':           'Turno',
    # Columna de área
    'col_area':            'Área',
    # Columna de razón de renuncia
    'col_motivo':          'Razón de Renuncia',
    # Columna de encuesta de salida
    'col_encuesta':        'Encuesta de salida 4FRH-209',
    # Columna de supervisor
    'col_supervisor':      'Supervisor',
    # Columna de monto de finiquito
    'col_finiquito':       'Monto Finiquito',
    # Columnas de fechas
    'col_fecha_baja':      'Fecha de baja en el Sistema',
    'col_fecha_alta':      'Fecha de Alta',
    'col_fecha_udt':       'Fecha de último día de trabajo (UDT)',
    'col_fecha_finiquito': 'Fecha en que se hizo el finiquito',
    'col_entrega_finiquito':'Fecha de entrega de finiquito',
    'col_ultimo_aumento':  'Último cambio de salario',
    'col_faltas_detalle':  ['Falta 1', 'Falta 2', 'Falta 3', 'Falta 4'],
    # Columnas a eliminar (identificadores)
    'cols_eliminar':       ['Empleado#', 'Nombre', 'Depto.',
                            'Número de semana de las últimas horas trabajadas'],
    # Columna de horas trabajadas
    'col_horas':           'Total de horas trabajadas  en la última semana',
    # Columna de entrenamiento
    'col_entrenamiento':   'Cumplió con periodo de entrenamiento',
    # Semanas para definir rotación temprana
    'semanas_temprana':    13,
    # Umbral de faltas para riesgo alto
    'umbral_faltas_alto':  4,
    # Umbral de días sin aumento para alerta
    'umbral_dias_aumento': 180,
}

# Palabras clave por tema en la encuesta de salida
TEMAS_ENCUESTA = {
    'Salario / Horas extras':   ['salario','sueldo','dinero','pago','horas extras','horas extra'],
    'Distancia / Transporte':   ['lejos','distancia','transporte','ciudad','camion','camión'],
    'Familia / Salud':          ['familia','mamá','mama','hijo','personal','salud','enferm'],
    'Ambiente laboral':         ['ambiente','compañeros','jefe','supervisor','trato'],
    'Prestaciones':             ['prestaciones','vales','comedor','seguro','beneficio'],
    'Recomendaría la empresa':  ['si recomend','sí recomend','muy bien','excelente',
                                 'si, porque','sí, porque'],
}

print("✅ Configuración lista")


In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("📂 Sube tu archivo Excel de rotación...")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df_raw = pd.read_excel(filename)
print(f"\n✅ Archivo cargado: {filename}")
print(f"   Registros: {len(df_raw)}")
print(f"   Columnas:  {len(df_raw.columns)}")
print(f"\n📋 Columnas encontradas:")
for i, col in enumerate(df_raw.columns, 1):
    print(f"   [{i:02d}] {col}")


---
## Sección 2 — Limpieza y procesamiento
> Se ejecuta automáticamente. No modificar.


In [ ]:
df = df_raw.copy()

# Parsear fechas (formato americano mm/dd/aaaa)
fecha_cols = [
    CONFIG['col_fecha_baja'], CONFIG['col_fecha_alta'], CONFIG['col_fecha_udt'],
    CONFIG['col_fecha_finiquito'], CONFIG['col_entrega_finiquito'],
    CONFIG['col_ultimo_aumento']
] + CONFIG['col_faltas_detalle']

for col in fecha_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=False)

# Derivar métricas desde fechas
df['dias_demora_finiquito'] = (
    df[CONFIG['col_entrega_finiquito']] - df[CONFIG['col_fecha_finiquito']]
).dt.days.fillna(0).astype(int)

df['dias_sin_aumento'] = (
    df[CONFIG['col_fecha_baja']] - df[CONFIG['col_ultimo_aumento']]
).dt.days.fillna(df[CONFIG['col_antiguedad']] * 7).astype(int)

df['dias_entre_baja_udt'] = (
    df[CONFIG['col_fecha_baja']] - df[CONFIG['col_fecha_udt']]
).dt.days.fillna(0).astype(int)

df['faltas_ultimo_mes'] = df[CONFIG['col_faltas_detalle']].notna().sum(axis=1)

# Categorizar motivo de salida
def cat_motivo(x):
    x = str(x).upper()
    if 'FALTA'      in x: return 'Baja por Faltas'
    if 'VOLUNTAR'   in x: return 'Renuncia Voluntaria'
    if 'FAMILIAR'   in x: return 'Familiar/Personal'
    if 'OPORTUNIDAD'in x: return 'Mejor Oportunidad'
    return 'Otro'

df['categoria_motivo']  = df[CONFIG['col_motivo']].apply(cat_motivo)
df['rotacion_temprana'] = (df[CONFIG['col_antiguedad']] < CONFIG['semanas_temprana']).astype(int)
df[CONFIG['col_area']]  = df[CONFIG['col_area']].fillna('Sin Área')

# Procesar encuesta de salida → columnas binarias
enc = df[CONFIG['col_encuesta']].fillna('').str.lower()
for tema, kws in TEMAS_ENCUESTA.items():
    col_name = f'enc_{tema[:15].strip().replace(" ","_").replace("/","_")}'
    df[col_name] = enc.apply(lambda x: 1 if any(k in x for k in kws) else 0)

temas_enc_cols = [f'enc_{t[:15].strip().replace(" ","_").replace("/","_")}' for t in TEMAS_ENCUESTA]

print("✅ Limpieza completada")
print(f"   Métricas derivadas de fechas: dias_demora_finiquito, dias_sin_aumento,")
print(f"   dias_entre_baja_udt, faltas_ultimo_mes")
print(f"   Temas de encuesta detectados: {len(temas_enc_cols)}")
print(f"\n📊 Distribución de motivos de salida:")
print(df['categoria_motivo'].value_counts().to_string())


---
## Sección 3 — PCA (Análisis de Componentes Principales)
> Identifica qué variables tienen más peso para describir la rotación


In [ ]:
# ── Sección 3 ──
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
# --- Preparar columnas a eliminar ---
cols_eliminar = [c for c in CONFIG['cols_eliminar'] if c in df.columns]
cols_fechas   = fecha_cols + CONFIG['col_faltas_detalle']
cols_texto    = [CONFIG['col_encuesta'], CONFIG['col_motivo'],
                 CONFIG['col_supervisor'], CONFIG['col_entrenamiento']]

df_pca = df.drop(columns=cols_eliminar + cols_fechas + cols_texto, errors='ignore')

# --- Encoding: entrenamiento binario ---
if CONFIG['col_entrenamiento'] in df.columns:
    df_pca = df_pca.copy()
    df_pca['entrenamiento'] = df[CONFIG['col_entrenamiento']].str.lower().map(
        {'si':1,'sí':1,'no':0}).fillna(0).astype(int)

# --- Encoding: supervisor por tasa de rotación ---
if CONFIG['col_supervisor'] in df.columns:
    sup_tasa = (df[CONFIG['col_supervisor']].value_counts() / len(df)).to_dict()
    df_pca['supervisor_tasa'] = df[CONFIG['col_supervisor']].map(sup_tasa).round(4)

# --- Encoding: categóricas con One-Hot ---
cat_cols_pca = [c for c in [CONFIG['col_area'],
                             'Tipo de baja en el Sistema',
                             'Razon capturada en Sistema',
                             'Puesto'] if c in df.columns]
df_cat = df[cat_cols_pca].copy()
for col in cat_cols_pca:
    df_cat[col] = df_cat[col].fillna('Desconocido').astype(str).str.strip()

dummies = pd.get_dummies(df_cat, drop_first=False).astype(int)
dummies.columns = (dummies.columns.str.lower()
                   .str.replace(' ', '_', regex=False)
                   .str.replace(r'[^a-z0-9_]', '', regex=True))

# --- Unir todo en un solo DataFrame numérico ---
df_pca = df_pca.drop(columns=[c for c in cat_cols_pca if c in df_pca.columns], errors='ignore')
enc_df = df[temas_enc_cols].copy()

df_pca = pd.concat([
    df_pca.reset_index(drop=True),
    dummies.reset_index(drop=True),
    enc_df.reset_index(drop=True)
], axis=1)

df_pca = df_pca.select_dtypes(include=[np.number]).fillna(0)

# --- Eliminar columnas duplicadas ---
df_pca = df_pca.loc[:, ~df_pca.columns.duplicated()].copy()

# --- Estandarizar con Z-score ---
nunique_series = df_pca.apply(lambda col: col.nunique(), axis=0)
bin_cols = nunique_series[nunique_series <= 2].index.tolist()
num_cols = nunique_series[nunique_series  > 2].index.tolist()

scaler = StandardScaler()
X = df_pca.copy()
if num_cols:
    X[num_cols] = scaler.fit_transform(df_pca[num_cols])
feature_names = X.columns.tolist()

# --- PCA ---
pca_full = PCA()
pca_full.fit(X.values)
var_ratio = pca_full.explained_variance_ratio_
var_acum  = np.cumsum(var_ratio)
n_comp    = max(int(np.argmax(var_acum >= 0.80)) + 1, 2)

pca_final = PCA(n_components=n_comp)
scores    = pca_final.fit_transform(X.values)
comp_lbls = [f'CP{i+1}' for i in range(n_comp)]

df_loadings = pd.DataFrame(
    pca_final.components_.T,
    index=feature_names,
    columns=comp_lbls
)

# --- Importancia ponderada ---
imp = (df_loadings**2).dot(pca_final.explained_variance_ratio_)
df_ranking = pd.DataFrame({
    'Variable':      feature_names,
    'Importancia_%': (imp / imp.sum() * 100).round(2)
})
df_ranking = df_ranking.sort_values('Importancia_%', ascending=False).reset_index(drop=True)
df_ranking['Acumulado_%'] = df_ranking['Importancia_%'].cumsum().round(2)
df_ranking.index += 1
n_vital = int(np.argmax(df_ranking['Acumulado_%'] >= 80)) + 1

# --- Gráfica: varianza explicada ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PCA — Varianza Explicada', fontsize=14, fontweight='bold', color='#1E3A5F')

ax1 = axes[0]
colors_bar = ['#2D6A4F' if v <= 80 else '#AAAAAA' for v in var_acum * 100]
ax1.bar(range(1, len(var_ratio)+1), var_ratio * 100, color=colors_bar, alpha=0.8)
ax1.plot(range(1, len(var_ratio)+1), var_acum * 100, 'o-', color='#922B21', linewidth=2)
ax1.axhline(80, color='#922B21', linestyle='--', alpha=0.5, label='80% umbral')
ax1.axvline(n_comp, color='#1E3A5F', linestyle='--', alpha=0.5, label=f'CP{n_comp} seleccionado')
ax1.set_xlabel('Componente Principal')
ax1.set_ylabel('Varianza %')
ax1.set_title('Varianza por Componente')
ax1.legend()
ax1.grid(alpha=0.3)

ax2 = axes[1]
top_vars   = df_ranking.head(15)
colors_v   = ['#1E3A5F' if i <= n_vital else '#BBBBBB' for i in range(1, len(top_vars)+1)]
ax2.barh(range(len(top_vars)), top_vars['Importancia_%'], color=colors_v)
ax2.set_yticks(range(len(top_vars)))
ax2.set_yticklabels([v[:30] for v in top_vars['Variable']], fontsize=8)
ax2.invert_yaxis()
ax2.set_xlabel('Importancia %')
ax2.set_title(f'Top 15 Variables — Zona vital: primeras {n_vital}')
ax2.axhline(n_vital - 0.5, color='#922B21', linestyle='--', linewidth=1.5, label='Corte 80%')
ax2.legend()
ax2.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"Variables totales:              {len(feature_names)}")
print(f"Componentes para 80% varianza: {n_comp}")
print(f"Variables zona vital (80/20):  {n_vital}")
print(f"\nTop 5 variables mas importantes:")
print(df_ranking[['Variable','Importancia_%','Acumulado_%']].head(5).to_string())

In [ ]:
# ── Gráfica 3: Heatmap de loadings ──
fig, ax = plt.subplots(figsize=(max(8, n_comp*2), min(20, len(feature_names)*0.4 + 2)))
top_n  = min(20, len(feature_names))
df_heat = df_loadings.head(top_n) if len(df_loadings) > top_n else df_loadings

# Ordenar por importancia
order = df_ranking.head(top_n)['Variable'].tolist()
df_heat = df_loadings.loc[[v for v in order if v in df_loadings.index]]

sns.heatmap(df_heat, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Loading (peso)'})
ax.set_title(f'Loadings — Top {top_n} variables por componente\n'
             f'Azul = carga positiva | Rojo = carga negativa | Más intenso = más peso',
             fontsize=11, fontweight='bold', color='#1E3A5F')
ax.set_xlabel('Componente Principal')
ax.set_ylabel('Variable')
plt.tight_layout()
plt.savefig('pca_loadings.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Heatmap de loadings generado")
print("\n💡 Cómo leer el heatmap:")
print("   Azul intenso  → variable aumenta cuando ese componente sube")
print("   Rojo intenso  → variable disminuye cuando ese componente sube")
print("   Blanco/neutro → variable no tiene relación con ese componente")


---
## Sección 4 — Análisis Estadístico por grupos
> Compara las variables clave del PCA entre grupos de empleados


In [ ]:
vars_clave = [CONFIG['col_faltas'], 'faltas_ultimo_mes', CONFIG['col_permisos'],
              CONFIG['col_antiguedad'], CONFIG['col_salario'],
              'dias_sin_aumento', CONFIG['col_horas'], CONFIG['col_finiquito']]
vars_clave = [v for v in vars_clave if v in df.columns]

col_turno = CONFIG['col_turno']
turnos    = sorted(df[col_turno].dropna().unique())

# ── Gráfica 4: Comparativo por turno ──
n_vars = len(vars_clave)
fig, axes = plt.subplots(2, (n_vars+1)//2, figsize=(16, 10))
axes = axes.flatten()
fig.suptitle('Análisis Estadístico — Variables clave por Turno',
             fontsize=13, fontweight='bold', color='#1E3A5F')

palette = ['#2E6B8A','#922B21','#2D6A4F','#7D3C08']
for i, var in enumerate(vars_clave):
    ax = axes[i]
    data_by_turno = [df[df[col_turno]==t][var].dropna().values for t in turnos]
    bp = ax.boxplot(data_by_turno, patch_artist=True, notch=False,
                    medianprops=dict(color='white', linewidth=2))
    for patch, color in zip(bp['boxes'], palette):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_xticklabels([f'Turno {t}' for t in turnos], fontsize=8)
    ax.set_title(var[:25], fontsize=9, fontweight='bold')
    ax.grid(alpha=0.3)
    # Añadir promedio
    for j, vals in enumerate(data_by_turno):
        if len(vals): ax.text(j+1, np.mean(vals), f'{np.mean(vals):.1f}',
                              ha='center', va='bottom', fontsize=7, color='#333333')

for i in range(n_vars, len(axes)): axes[i].set_visible(False)
plt.tight_layout()
plt.savefig('estadistico_turno.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla resumen
print("📊 Promedio de variables clave por Turno:")
resumen_turno = df.groupby(col_turno)[vars_clave].mean().round(2)
resumen_turno.index = [f'Turno {t}' for t in resumen_turno.index]
print(resumen_turno.to_string())


In [ ]:
# ── Gráfica 5: Rotación temprana vs tardía ──
df['grupo_antiguedad'] = df[CONFIG['col_antiguedad']].apply(
    lambda x: f'Temprana\n(< {CONFIG["semanas_temprana"]} sem)'
    if x < CONFIG['semanas_temprana'] else f'Tardía\n(≥ {CONFIG["semanas_temprana"]} sem)'
)

fig, axes = plt.subplots(1, min(4, len(vars_clave)), figsize=(16, 5))
if len(vars_clave) == 1: axes = [axes]
fig.suptitle('Rotación Temprana vs Tardía — Variables del PCA',
             fontsize=13, fontweight='bold', color='#1E3A5F')

grupos     = df['grupo_antiguedad'].unique()
cols_show  = vars_clave[:4]
for i, var in enumerate(cols_show):
    ax  = axes[i]
    data = [df[df['grupo_antiguedad']==g][var].dropna().values for g in grupos]
    bp  = ax.boxplot(data, patch_artist=True,
                     medianprops=dict(color='white', linewidth=2))
    colors_g = ['#922B21','#2D6A4F']
    for patch, color in zip(bp['boxes'], colors_g):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_xticklabels(grupos, fontsize=8)
    ax.set_title(var[:22], fontsize=9, fontweight='bold')
    ax.grid(alpha=0.3)
    for j, vals in enumerate(data):
        if len(vals): ax.text(j+1, np.mean(vals), f'x̄={np.mean(vals):.1f}',
                              ha='center', va='bottom', fontsize=8, color='#333')

plt.tight_layout()
plt.savefig('estadistico_antiguedad.png', dpi=150, bbox_inches='tight')
plt.show()

n_temp = len(df[df['rotacion_temprana']==1])
n_tard = len(df[df['rotacion_temprana']==0])
print(f"✅ Rotación Temprana (< {CONFIG['semanas_temprana']} sem): {n_temp} empleados ({n_temp/len(df)*100:.0f}%)")
print(f"   Rotación Tardía  (≥ {CONFIG['semanas_temprana']} sem): {n_tard} empleados ({n_tard/len(df)*100:.0f}%)")


---
## Sección 5 — Pareto 80/20
> Identifica el 20% de causas que genera el 80% de las bajas


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Análisis Pareto 80/20 — 3 Dimensiones', fontsize=14,
             fontweight='bold', color='#1E3A5F')

def plot_pareto(ax, data_series, title, color_vital, color_trivial):
    counts = data_series.value_counts().reset_index()
    counts.columns = ['Cat','N']
    counts['%']    = (counts['N']/len(data_series)*100).round(1)
    counts['Acum'] = counts['%'].cumsum().round(1)
    n_vital_p      = int(np.argmax(counts['Acum'] >= 80)) + 1
    colors         = [color_vital if i < n_vital_p else color_trivial
                      for i in range(len(counts))]
    bars = ax.bar(range(len(counts)), counts['%'], color=colors, alpha=0.85, zorder=2)
    ax2  = ax.twinx()
    ax2.plot(range(len(counts)), counts['Acum'], 'o-', color='#922B21',
             linewidth=2, zorder=3, label='% Acumulado')
    ax2.axhline(80, color='#922B21', linestyle='--', alpha=0.4)
    ax2.set_ylabel('% Acumulado', color='#922B21')
    ax2.set_ylim(0, 115)
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels([str(c)[:18] for c in counts['Cat']],
                       rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('% del Total')
    ax.set_title(title, fontweight='bold', color='#1E3A5F', fontsize=10)
    ax.grid(alpha=0.3, axis='y', zorder=1)
    # Etiquetas
    for bar, pct in zip(bars, counts['%']):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{pct}%', ha='center', va='bottom', fontsize=8)
    return counts, n_vital_p

# Dimensión 1 — Motivo
mot_data, n_v1 = plot_pareto(axes[0], df['categoria_motivo'],
                              'Dim 1: Motivo de Salida', '#2E6B8A', '#AAAAAA')
# Dimensión 2 — Turno
tur_data, n_v2 = plot_pareto(axes[1], df[CONFIG['col_turno']].astype(str).apply(lambda x: f'Turno {x}'),
                              'Dim 2: Turno', '#2D6A4F', '#AAAAAA')
# Dimensión 3 — Área
are_data, n_v3 = plot_pareto(axes[2], df[CONFIG['col_area']],
                              'Dim 3: Área', '#7D3C08', '#AAAAAA')

plt.tight_layout()
plt.savefig('pareto_3dim.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Pareto 80/20 calculado")
print(f"\n📊 Dimensión 1 — Motivo:")
print(mot_data.to_string(index=False))
print(f"\n📊 Dimensión 2 — Turno:")
print(tur_data.to_string(index=False))
print(f"\n📊 Dimensión 3 — Área:")
print(are_data.to_string(index=False))


---
## Sección 6 — Causa Raíz (5 Porqués con datos)
> Cruza PCA + Pareto + Encuesta para encontrar el origen de la rotación


In [ ]:
# ── Gráfica 6: Encuesta por turno ──
fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle('Encuesta de Salida — Temas mencionados por Turno',
             fontsize=13, fontweight='bold', color='#1E3A5F')

enc_turno = df.groupby(CONFIG['col_turno'])[temas_enc_cols].sum()
x         = np.arange(len(temas_enc_cols))
width     = 0.8 / len(enc_turno)
colors_t  = ['#2E6B8A','#922B21','#2D6A4F','#7D3C08']

for i, (turno, row_data) in enumerate(enc_turno.iterrows()):
    offset = (i - len(enc_turno)/2 + 0.5) * width
    bars   = ax.bar(x + offset, row_data.values, width*0.9,
                    label=f'Turno {turno}', color=colors_t[i % len(colors_t)], alpha=0.8)

ax.set_xticks(x)
temas_short = [list(TEMAS_ENCUESTA.keys())[i][:20] for i in range(len(temas_enc_cols))]
ax.set_xticklabels(temas_short, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('N empleados que lo mencionaron')
ax.legend(title='Turno')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('encuesta_turno.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Tabla de causa raíz automática ──
print("=" * 65)
print("🔍 ANÁLISIS DE CAUSA RAÍZ — 5 PORQUÉS CON DATOS")
print("=" * 65)

turno_stats = df.groupby(CONFIG['col_turno']).agg(
    N               = (CONFIG['col_faltas'], 'count'),
    avg_faltas      = (CONFIG['col_faltas'], 'mean'),
    avg_dias_aumento= ('dias_sin_aumento', 'mean'),
    avg_salario     = (CONFIG['col_salario'], 'mean'),
    avg_permisos    = (CONFIG['col_permisos'], 'mean'),
).round(2)

turno_max_rot = turno_stats['N'].idxmax()
turno_min_rot = turno_stats['N'].idxmin()

t_max = turno_stats.loc[turno_max_rot]
t_min = turno_stats.loc[turno_min_rot]

motivo_top = mot_data.iloc[0]['Cat']
motivo_pct = mot_data.iloc[0]['%']
area_top   = are_data.iloc[0]['Cat']
var_top1   = df_ranking.iloc[0]['Variable']
var_top2   = df_ranking.iloc[1]['Variable']

sep = "=" * 65
print(sep)
print("SINTOMA - QUE PASA:")
print(f"  {len(df)} bajas registradas")
print(f"  Turno {turno_max_rot} concentra {t_max['N']:.0f} de {len(df)} bajas ({t_max['N']/len(df)*100:.0f}%)")
print(f"  Motivo mas frecuente: {motivo_top} = {motivo_pct}% de los casos")
print()
print("POR QUE 1 - Motivo registrado:")
print(f"  El {motivo_pct:.0f}% son '{motivo_top}'")
print(f"  El empleado no renuncia: acumula faltas hasta que el sistema lo da de baja")
print(f"  Es una desconexion progresiva, no una decision activa")
print()
print("POR QUE 2 - Comportamiento observable:")
print(f"  Turno {turno_max_rot}: {t_max['avg_faltas']:.1f} faltas promedio | {t_max['avg_permisos']:.1f} permisos promedio")
print(f"  Turno {turno_min_rot}: {t_min['avg_faltas']:.1f} faltas promedio | {t_min['avg_permisos']:.1f} permisos promedio")
print(f"  El Turno {turno_max_rot} falta mas y NO pide permiso - senal de desconexion")
print()
print(f"POR QUE 3 - Condiciones laborales (voz del empleado):")
print(f"  Temas mencionados por Turno {turno_max_rot} en encuesta de salida:")

enc_max = df[df[CONFIG['col_turno']]==turno_max_rot][temas_enc_cols].sum()
for tema_col, tema_label in zip(temas_enc_cols, TEMAS_ENCUESTA.keys()):
    v = int(enc_max[tema_col])
    if v > 0:
        print(f"    - {tema_label}: {v} empleado(s) lo mencionaron")

print()
print("POR QUE 4 - Condicion estructural:")
print(f"  Turno {turno_max_rot}: {t_max['avg_dias_aumento']:.0f} dias sin aumento | Salario: ${t_max['avg_salario']:.2f}")
print(f"  Turno {turno_min_rot}: {t_min['avg_dias_aumento']:.0f} dias sin aumento | Salario: ${t_min['avg_salario']:.2f}")
print(f"  Sin diferencial salarial entre turnos a pesar de distinta carga operativa")
print()
print("CAUSA RAIZ IDENTIFICADA:")
print(f"  Variables criticas del PCA: {var_top1} y {var_top2}")
print(f"  Turno {turno_max_rot}: mayor carga logistica + sin compensacion diferenciada")
print(f"  + politica de permisos no adaptada = ausentismo progresivo -> baja por faltas")
print(sep)


---
## Sección 7 — Exportar reporte de resultados
> Genera el archivo Excel y HTML con todos los análisis y lo descarga automáticamente


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from google.colab import files

# --- Colores ---
AZUL  = 'FF1E3A5F'; TEAL = 'FF2E6B8A'; VERDE = 'FF2D6A4F'
ROJO  = 'FF922B21'; NAR  = 'FF7D3C08'; WHITE = 'FFFFFFFF'
RC    = 'FFFDEDEC'; GC   = 'FFE9F7EF'; AC    = 'FFFFF3CD'; BC = 'FFE8F4FD'
thin  = Side(style='thin', color='FFB0B0B0')
borde = Border(left=thin, right=thin, top=thin, bottom=thin)

def hc(ws, r, c, v, bg, fc='FFFFFFFF', bold=True, sz=10):
    cel = ws.cell(row=r, column=c, value=v)
    cel.font      = Font(bold=bold, color=fc, name='Arial', size=sz)
    cel.fill      = PatternFill('solid', start_color=bg)
    cel.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cel.border    = borde
    return cel

def dc(ws, r, c, v, fmt=None, bold=False, bg=WHITE, center=True, color='FF000000', wrap=False):
    cel = ws.cell(row=r, column=c, value=v)
    cel.font      = Font(name='Arial', size=9, bold=bold, color=color)
    cel.border    = borde
    cel.alignment = Alignment(horizontal='center' if center else 'left',
                              vertical='center', wrap_text=wrap)
    if fmt: cel.number_format = fmt
    if bg:  cel.fill = PatternFill('solid', start_color=bg)
    return cel

def titulo_hoja(ws, t, sub, bg, n):
    ws.row_dimensions[1].height = 28
    ws.row_dimensions[2].height = 15
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=n)
    c = ws.cell(row=1, column=1, value=t)
    c.font = Font(bold=True, color='FFFFFFFF', name='Arial', size=13)
    c.fill = PatternFill('solid', start_color=bg)
    c.alignment = Alignment(horizontal='center', vertical='center')
    ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=n)
    c2 = ws.cell(row=2, column=1, value=sub)
    c2.font = Font(color='FF555555', name='Arial', size=9, italic=True)
    c2.fill = PatternFill('solid', start_color='FFF5F5F5')
    c2.alignment = Alignment(horizontal='center', vertical='center')

def set_widths(ws, widths):
    for i, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = w

wb = Workbook()
wb.remove(wb.active)

# ── Hoja 1: Ranking PCA ──
ws = wb.create_sheet('1_PCA_Variables')
titulo_hoja(ws, 'PCA - RANKING DE VARIABLES',
            'Zona vital: ' + str(n_vital) + ' variables = 80% de la rotacion', AZUL, 5)
for j, h in enumerate(['#', 'Variable', 'Importancia %', 'Acumulado %', 'Zona Pareto'], 1):
    hc(ws, 3, j, h, AZUL)
for i, row in df_ranking.iterrows():
    acum = row['Acumulado_%']
    bg   = GC if acum <= 80 else ('FFF5F5F5' if acum <= 90 else WHITE)
    bold = acum <= 80
    dc(ws, i+3, 1, i, bg=bg, bold=bold)
    dc(ws, i+3, 2, row['Variable'], bg=bg, bold=bold, center=False)
    dc(ws, i+3, 3, row['Importancia_%'], '0.00"%"', bg=bg, bold=bold)
    dc(ws, i+3, 4, acum/100, '0.00%', bg=bg, bold=bold,
       color='FF1A6B3C' if acum <= 80 else ('FF9A6B00' if acum <= 90 else 'FF888888'))
    dc(ws, i+3, 5, 'VITAL' if acum <= 80 else 'Trivial', bg=bg, bold=bold,
       color='FF1A6B3C' if acum <= 80 else 'FF9A6B00')
set_widths(ws, [5, 38, 14, 14, 12])
ws.freeze_panes = 'A4'

# ── Hoja 2: Loadings ──
ws2 = wb.create_sheet('2_PCA_Loadings')
n_cols_h2 = n_comp + 1
titulo_hoja(ws2, 'LOADINGS - PESO POR COMPONENTE',
            'Azul intenso = carga alta positiva | Rojo intenso = carga alta negativa',
            TEAL, n_cols_h2)
hc(ws2, 3, 1, 'Variable', TEAL)
for j, cp in enumerate(comp_lbls, 2):
    vp    = round(pca_final.explained_variance_ratio_[j-2] * 100, 1)
    label = cp + ' (' + str(vp) + '% var)'
    hc(ws2, 3, j, label, TEAL)
for i, var in enumerate(df_loadings.index, 1):
    bg = BC if i % 2 == 0 else WHITE
    dc(ws2, i+3, 1, var, bg=bg, center=False)
    for j, cp in enumerate(comp_lbls, 2):
        val = float(df_loadings.loc[var, cp])
        c   = dc(ws2, i+3, j, round(val, 4), '0.0000', bg=bg)
        if abs(val) >= 0.5:
            c.font = Font(bold=True,
                          color=ROJO if val < 0 else AZUL,
                          name='Arial', size=9)
        elif abs(val) >= 0.3:
            c.font = Font(bold=True,
                          color='FF784212' if val < 0 else 'FF1E8BC3',
                          name='Arial', size=9)
set_widths(ws2, [30] + [16] * n_comp)
ws2.freeze_panes = 'A4'

# ── Hoja 3: Pareto 3 dimensiones ──
ws3 = wb.create_sheet('3_Pareto_80_20')
titulo_hoja(ws3, 'PARETO 80/20 - 3 DIMENSIONES',
            'Verde = zona vital | Gris = zona trivial', VERDE, 6)

def write_pareto_section(ws, start_row, data, title, bg_hdr):
    ws.row_dimensions[start_row].height = 16
    ws.merge_cells(start_row=start_row, start_column=1,
                   end_row=start_row, end_column=6)
    c = ws.cell(row=start_row, column=1, value='  ' + title)
    c.font = Font(bold=True, color='FFFFFFFF', name='Arial', size=10)
    c.fill = PatternFill('solid', start_color=bg_hdr)
    c.alignment = Alignment(horizontal='left', vertical='center')
    for j, h in enumerate(['#', 'Categoria', 'N Bajas', '% Total', 'Acumulado %', 'Zona'], 1):
        hc(ws, start_row+1, j, h, bg_hdr)
    for i, row in data.iterrows():
        acum = row['Acum']
        bold = acum <= 80
        bg   = GC if acum <= 80 else WHITE
        dc(ws, start_row+1+i, 1, i+1,           bg=bg, bold=bold)
        dc(ws, start_row+1+i, 2, str(row['Cat']),bg=bg, bold=bold, center=False)
        dc(ws, start_row+1+i, 3, int(row['N']), bg=bg, bold=bold)
        dc(ws, start_row+1+i, 4, row['%']/100,  '0.0%', bg=bg, bold=bold)
        dc(ws, start_row+1+i, 5, acum/100,      '0.0%', bg=bg, bold=bold,
           color='FF1A6B3C' if acum <= 80 else 'FF888888')
        dc(ws, start_row+1+i, 6, 'VITAL' if acum <= 80 else 'Trivial',
           bg=bg, bold=bold,
           color='FF1A6B3C' if acum <= 80 else 'FF9A6B00')
    return start_row + len(data) + 4

r = write_pareto_section(ws3, 3,  mot_data, 'DIMENSION 1 - Motivo de Salida', VERDE)
r = write_pareto_section(ws3, r,  tur_data, 'DIMENSION 2 - Turno', TEAL)
r = write_pareto_section(ws3, r,  are_data, 'DIMENSION 3 - Area', NAR)
set_widths(ws3, [5, 26, 10, 12, 14, 12])
ws3.freeze_panes = 'A4'

# ── Hoja 4: Estadístico por turno ──
ws4 = wb.create_sheet('4_Estadistico_Turno')
n_cols_h4 = len(vars_clave) + 2
titulo_hoja(ws4, 'ANALISIS ESTADISTICO - VARIABLES POR TURNO',
            'Variables criticas del PCA cruzadas por turno', ROJO, n_cols_h4)
hc(ws4, 3, 1, 'Turno', ROJO)
hc(ws4, 3, 2, 'N Bajas', ROJO)
for j, v in enumerate(vars_clave, 3):
    hc(ws4, 3, j, v[:20], ROJO)
for i, (turno, row_data) in enumerate(turno_stats.iterrows(), 1):
    bg   = RC if turno == turno_max_rot else GC
    bold = turno == turno_max_rot
    dc(ws4, i+3, 1, 'Turno ' + str(turno), bold=bold, bg=bg,
       color=ROJO if turno == turno_max_rot else VERDE)
    dc(ws4, i+3, 2, int(row_data['N']), bg=bg, bold=bold)
    for j, v in enumerate(vars_clave, 3):
        val = df[df[CONFIG['col_turno']] == turno][v].mean()
        dc(ws4, i+3, j, round(float(val), 2) if pd.notna(val) else 0,
           '0.00', bg=bg)
set_widths(ws4, [12, 10] + [20] * len(vars_clave))
ws4.freeze_panes = 'A4'

# ── Hoja 5: Causa Raíz ──
ws5 = wb.create_sheet('5_Causa_Raiz')
titulo_hoja(ws5, 'CAUSA RAIZ - 5 PORQUES CON DATOS',
            'Cada porque respaldado con evidencia del PCA + Pareto + Encuesta', AZUL, 4)

var_top1 = df_ranking.iloc[0]['Variable']
var_top2 = df_ranking.iloc[1]['Variable']

filas_cr = [
    ('SINTOMA - QUE PASA', RC, ROJO,
     str(len(df)) + ' bajas — Turno ' + str(turno_max_rot) +
     ' concentra ' + str(int(t_max['N'])) +
     ' (' + str(round(t_max['N']/len(df)*100)) + '%)',
     'Pareto Dim. 2 y 3'),
    ('POR QUE 1 - Motivo registrado', AC, NAR,
     str(round(motivo_pct)) + '% son "' + str(motivo_top) +
     '" — desconexion progresiva, no renuncia activa',
     'Pareto Dim. 1'),
    ('POR QUE 2 - Comportamiento', AC, NAR,
     'Turno ' + str(turno_max_rot) + ': ' +
     str(round(t_max['avg_faltas'], 1)) + ' faltas, ' +
     str(round(t_max['avg_permisos'], 1)) + ' permisos — no avisa, solo falta',
     'Cruce Turno x PCA'),
    ('POR QUE 3 - Condiciones laborales', AC, NAR,
     'Empleados del Turno ' + str(turno_max_rot) +
     ' mencionan en encuesta: distancia, transporte, condiciones del turno',
     'Encuesta x Turno'),
    ('POR QUE 4 - Condicion estructural', RC, ROJO,
     'Turno ' + str(turno_max_rot) + ': ' +
     str(round(t_max['avg_dias_aumento'])) + ' dias sin aumento vs ' +
     str(round(t_min['avg_dias_aumento'])) + ' del Turno ' +
     str(turno_min_rot) + ' — sin diferencial salarial',
     'PCA CP1 + Estadistico'),
    ('CAUSA RAIZ IDENTIFICADA', GC, VERDE,
     'Turno ' + str(turno_max_rot) +
     ' sin compensacion diferenciada + ' +
     str(round(t_max['avg_dias_aumento'])) +
     ' dias sin aumento + politica de permisos rigida = ausentismo -> baja por faltas',
     'PCA + Pareto + Encuesta'),
]

r5 = 3
for nivel, bg, color, resp, fuente in filas_cr:
    ws5.row_dimensions[r5].height   = 18
    ws5.row_dimensions[r5+1].height = 45
    ws5.merge_cells(start_row=r5, start_column=1, end_row=r5, end_column=4)
    c = ws5.cell(row=r5, column=1, value=nivel)
    c.font = Font(bold=True, color='FFFFFFFF', name='Arial', size=10)
    c.fill = PatternFill('solid', start_color=color)
    c.alignment = Alignment(horizontal='left', vertical='center')
    dc(ws5, r5+1, 1, nivel, bold=True, bg=bg, color=color)
    ws5.merge_cells(start_row=r5+1, start_column=2, end_row=r5+1, end_column=3)
    c2 = ws5.cell(row=r5+1, column=2, value=resp)
    c2.font      = Font(name='Arial', size=9, color='FF222222')
    c2.fill      = PatternFill('solid', start_color=bg)
    c2.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
    c2.border    = borde
    dc(ws5, r5+1, 4, 'Fuente: ' + fuente, bg='FFF5F5F5',
       center=False, color='FF555555', wrap=True)
    r5 += 3

set_widths(ws5, [22, 38, 20, 24])
ws5.freeze_panes = 'A3'

# --- Guardar y descargar ---
output_name = 'Analisis_Rotacion_Completo.xlsx'
wb.save(output_name)
files.download(output_name)

print('Excel descargado: ' + output_name)
print('Hojas generadas:')
print('  1_PCA_Variables    — Ranking de variables por importancia')
print('  2_PCA_Loadings     — Pesos por componente')
print('  3_Pareto_80_20     — 3 dimensiones Pareto')
print('  4_Estadistico_Turno— Variables clave por turno')
print('  5_Causa_Raiz       — 5 Porques con datos')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELDA DE NORMALIZACIÓN — Ejecutar ANTES del bloque de reportes
# Estandariza los nombres de columnas del Pareto
# ══════════════════════════════════════════════════════════════════

def normalizar_pareto(df_pareto):
    # Renombrar columnas a minúsculas consistentes
    col_map = {}
    for col in df_pareto.columns:
        col_lower = col.lower().strip()
        if col_lower in ['cat', 'motivo', 'turno', 'area']:
            col_map[col] = 'Cat'
        elif col_lower in ['n', 'count', 'bajas']:
            col_map[col] = 'N'
        elif col_lower in ['%', 'pct', 'porcentaje']:
            col_map[col] = 'pct'
        elif col_lower in ['acum', 'acumulado', 'acumulado_%', 'acum_%']:
            col_map[col] = 'acum'
    return df_pareto.rename(columns=col_map)

mot_data = normalizar_pareto(mot_data)
tur_data = normalizar_pareto(tur_data)
are_data = normalizar_pareto(are_data)

# Verificar que turno_stats tiene la columna avg_dias
if 'avg_dias_aumento' in turno_stats.columns and 'avg_dias' not in turno_stats.columns:
    turno_stats = turno_stats.rename(columns={'avg_dias_aumento': 'avg_dias'})

# Refrescar t_max y t_min
t_max = turno_stats.loc[turno_max_rot]
t_min = turno_stats.loc[turno_min_rot]

print('Columnas mot_data:', mot_data.columns.tolist())
print('Columnas tur_data:', tur_data.columns.tolist())
print('Columnas are_data:', are_data.columns.tolist())
print('Columnas turno_stats:', turno_stats.columns.tolist())
print()
print('Todo normalizado — ahora ejecuta el bloque de reportes.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELDA DE VERIFICACION — Ejecutar ANTES del bloque de reportes
# Confirma que todas las variables necesarias estan disponibles
# ══════════════════════════════════════════════════════════════════

variables_necesarias = {
    'var_ratio':     'Seccion 3 — PCA',
    'pca_final':     'Seccion 3 — PCA',
    'df_ranking':    'Seccion 3 — PCA',
    'df_loadings':   'Seccion 3 — PCA',
    'n_comp':        'Seccion 3 — PCA',
    'n_vital':       'Seccion 3 — PCA',
    'feature_names': 'Seccion 3 — PCA',
    'scores':        'Seccion 3 — PCA',
    'comp_lbls':     'Seccion 3 — PCA',
    'mot_data':      'Seccion 5 — Pareto',
    'tur_data':      'Seccion 5 — Pareto',
    'are_data':      'Seccion 5 — Pareto',
    'turno_stats':   'Seccion 6 — Causa Raiz',
    'turno_max_rot': 'Seccion 6 — Causa Raiz',
    'turno_min_rot': 'Seccion 6 — Causa Raiz',
    't_max':         'Seccion 6 — Causa Raiz',
    't_min':         'Seccion 6 — Causa Raiz',
    'df':            'Seccion 2 — Procesamiento',
    'CONFIG':        'Seccion 1 — Configuracion',
}

faltantes = []
for var, origen in variables_necesarias.items():
    try:
        eval(var)
        print('OK  ' + var)
    except NameError:
        print('FALTA  ' + var + '  <-  corre ' + origen)
        faltantes.append(var)

print()
if faltantes:
    print('ERROR: Faltan ' + str(len(faltantes)) + ' variables.')
    print('Corre las secciones indicadas arriba y vuelve a intentar.')
else:
    print('Todo listo — puedes ejecutar el bloque de reportes.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELDA DE REPORTES — Ejecutar cuando quieras, independiente
# Genera: Informe Ejecutivo + Informe Tecnico (ambos HTML)
# Para PDF: abre el HTML en Chrome -> Ctrl+P -> Guardar como PDF
# Requiere haber corrido antes las Secciones 1 a 6
# ══════════════════════════════════════════════════════════════════

import base64, io, datetime
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
from google.colab import files

hoy          = datetime.datetime.now().strftime('%d/%m/%Y')
hoy_archivo  = datetime.datetime.now().strftime('%Y%m%d')

# ─────────────────────────────────────────────
# HELPER: figura a base64
# ─────────────────────────────────────────────
def fig_to_b64(fig, dpi=130):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    plt.close(fig)
    return b64

def img_tag(b64, width='100%'):
    return '<img src="data:image/png;base64,' + b64 + '" style="width:' + width + ';border-radius:6px;"/>'

# ─────────────────────────────────────────────
# FIGURA 1 — Varianza PCA
# ─────────────────────────────────────────────
fig1, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig1.patch.set_facecolor('white')

ax1 = axes[0]
colors_b = ['#2D6A4F' if v <= 80 else '#CCCCCC' for v in np.cumsum(var_ratio)*100]
ax1.bar(range(1, len(var_ratio)+1), var_ratio*100, color=colors_b, alpha=0.85)
ax1.plot(range(1, len(var_ratio)+1), np.cumsum(var_ratio)*100,
         'o-', color='#922B21', linewidth=2, markersize=5)
ax1.axhline(80, color='#922B21', linestyle='--', alpha=0.4, linewidth=1)
ax1.set_xlabel('Componente Principal', fontsize=9)
ax1.set_ylabel('Varianza (%)', fontsize=9)
ax1.set_title('Varianza explicada por componente', fontsize=10, fontweight='bold')
ax1.grid(alpha=0.25, axis='y')

ax2 = axes[1]
top15   = df_ranking.head(15)
cols_v  = ['#1E3A5F' if i <= n_vital else '#CCCCCC' for i in range(1, len(top15)+1)]
ax2.barh(range(len(top15)), top15['Importancia_%'], color=cols_v, alpha=0.85)
ax2.set_yticks(range(len(top15)))
ax2.set_yticklabels([v[:28] for v in top15['Variable']], fontsize=7.5)
ax2.invert_yaxis()
ax2.axhline(n_vital - 0.5, color='#922B21', linestyle='--', linewidth=1.5)
ax2.set_xlabel('Importancia (%)', fontsize=9)
ax2.set_title('Top 15 variables — Zona vital: ' + str(n_vital), fontsize=10, fontweight='bold')
ax2.grid(alpha=0.25, axis='x')
legend_el = [mpatches.Patch(color='#1E3A5F', label='Zona vital (80%)'),
             mpatches.Patch(color='#CCCCCC', label='Zona trivial')]
ax2.legend(handles=legend_el, fontsize=8, loc='lower right')
fig1.tight_layout(pad=1.5)
b64_pca = fig_to_b64(fig1)

# ─────────────────────────────────────────────
# FIGURA 2 — Pareto 3 dimensiones
# ─────────────────────────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(14, 4.5))
fig2.patch.set_facecolor('white')
fig2.suptitle('Pareto 80/20 — 3 Dimensiones', fontsize=11,
              fontweight='bold', color='#1E3A5F')

def plot_pareto_ax(ax, data, title, color_vital, color_trivial):
    acum = data['acum'].values
    pcts = data['pct'].values
    cats = data['Cat'].astype(str).values
    n_v  = int(np.argmax(acum >= 80)) + 1
    cols = [color_vital if i < n_v else color_trivial for i in range(len(cats))]
    ax.bar(range(len(cats)), pcts, color=cols, alpha=0.85)
    ax2b = ax.twinx()
    ax2b.plot(range(len(cats)), acum, 'o-', color='#922B21', linewidth=2, markersize=5)
    ax2b.axhline(80, color='#922B21', linestyle='--', alpha=0.35, linewidth=1)
    ax2b.set_ylim(0, 115)
    ax2b.set_ylabel('% Acumulado', fontsize=8, color='#922B21')
    ax2b.tick_params(colors='#922B21', labelsize=7)
    ax.set_xticks(range(len(cats)))
    ax.set_xticklabels([c[:16] for c in cats], rotation=30, ha='right', fontsize=7.5)
    ax.set_ylabel('% del Total', fontsize=8)
    ax.set_title(title, fontsize=9, fontweight='bold', color='#1E3A5F')
    ax.grid(alpha=0.2, axis='y')
    for i, p in enumerate(pcts):
        ax.text(i, p + 0.5, str(p) + '%', ha='center', va='bottom', fontsize=7)

plot_pareto_ax(axes2[0], mot_data, 'Motivo de Salida', '#2E6B8A', '#AAAAAA')
plot_pareto_ax(axes2[1], tur_data, 'Turno',            '#2D6A4F', '#AAAAAA')
plot_pareto_ax(axes2[2], are_data, 'Area',             '#7D3C08', '#AAAAAA')
fig2.tight_layout(pad=1.5)
b64_pareto = fig_to_b64(fig2)

# ─────────────────────────────────────────────
# FIGURA 3 — Heatmap de loadings
# ─────────────────────────────────────────────
top_n   = min(15, len(feature_names))
order   = df_ranking.head(top_n)['Variable'].tolist()
df_heat = df_loadings.loc[[v for v in order if v in df_loadings.index]]
fig3, ax3 = plt.subplots(figsize=(max(7, n_comp*1.8), top_n * 0.48 + 1.5))
fig3.patch.set_facecolor('white')
sns.heatmap(df_heat, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax3, linewidths=0.4,
            cbar_kws={'label': 'Loading', 'shrink': 0.8})
ax3.set_title('Loadings PCA — Top ' + str(top_n) + ' variables',
              fontsize=10, fontweight='bold', color='#1E3A5F', pad=8)
ax3.set_xlabel('Componente Principal', fontsize=9)
ax3.set_ylabel('')
fig3.tight_layout()
b64_loadings = fig_to_b64(fig3)

# ─────────────────────────────────────────────
# FIGURA 4 — Variables clave por turno
# ─────────────────────────────────────────────
vars_graf = [v for v in [CONFIG['col_faltas'], 'dias_sin_aumento',
                          CONFIG['col_permisos'], CONFIG['col_salario']]
             if v in df.columns]
turnos_uniq = sorted(df[CONFIG['col_turno']].dropna().unique())
fig4, axes4 = plt.subplots(1, len(vars_graf), figsize=(13, 4))
fig4.patch.set_facecolor('white')
fig4.suptitle('Variables clave por Turno', fontsize=11,
              fontweight='bold', color='#1E3A5F')
pal = ['#2E6B8A', '#922B21', '#2D6A4F', '#7D3C08']
for i, var in enumerate(vars_graf):
    ax = axes4[i]
    data_t = [df[df[CONFIG['col_turno']]==t][var].dropna().values for t in turnos_uniq]
    bp = ax.boxplot(data_t, patch_artist=True,
                    medianprops=dict(color='white', linewidth=2))
    for patch, color in zip(bp['boxes'], pal):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    ax.set_xticklabels(['T' + str(t) for t in turnos_uniq], fontsize=9)
    ax.set_title(var[:22], fontsize=9, fontweight='bold')
    ax.grid(alpha=0.25, axis='y')
    for j, vals in enumerate(data_t):
        if len(vals):
            ax.text(j+1, float(np.mean(vals)),
                    str(round(float(np.mean(vals)), 1)),
                    ha='center', va='bottom', fontsize=8, color='#333')
fig4.tight_layout(pad=1.5)
b64_turno = fig_to_b64(fig4)

# ─────────────────────────────────────────────
# DATOS para tablas
# ─────────────────────────────────────────────
motivo_top1     = str(mot_data.iloc[0]['Cat'])
motivo_top1_pct = float(mot_data.iloc[0]['pct'])
motivo_top2     = str(mot_data.iloc[1]['Cat']) if len(mot_data) > 1 else ''
turno_max_pct   = round(t_max['N'] / len(df) * 100)
var1 = df_ranking.iloc[0]['Variable']
var2 = df_ranking.iloc[1]['Variable']
var3 = df_ranking.iloc[2]['Variable']

# ─────────────────────────────────────────────
# CSS COMPARTIDO
# ─────────────────────────────────────────────
CSS = """
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: Arial, sans-serif; font-size: 10pt; color: #222; background: #fff; }
h1 { font-size: 18pt; color: #1E3A5F; margin-bottom: 4px; }
h2 { font-size: 13pt; color: #1E3A5F; border-bottom: 2px solid #1E3A5F;
     padding-bottom: 4px; margin: 18px 0 10px; }
h3 { font-size: 10pt; color: #2E6B8A; margin: 14px 0 6px; }
p  { line-height: 1.55; margin-bottom: 8px; }
.page { max-width: 820px; margin: 0 auto; padding: 32px 40px; }
.header-box { background: #1E3A5F; color: white; padding: 20px 28px;
              border-radius: 8px; margin-bottom: 24px; }
.header-box h1 { color: white; font-size: 20pt; }
.header-box .sub { color: #B0C4D8; font-size: 9pt; margin-top: 6px; }
.kpi-row { display: flex; gap: 12px; margin: 16px 0; flex-wrap: wrap; }
.kpi { flex: 1; min-width: 140px; background: #f0f5fa;
       border-left: 4px solid #1E3A5F; padding: 12px 14px; border-radius: 4px; }
.kpi .num { font-size: 22pt; font-weight: bold; color: #1E3A5F; }
.kpi .lbl { font-size: 8pt; color: #555; margin-top: 2px; }
.kpi.rojo { border-color: #922B21; }
.kpi.rojo .num { color: #922B21; }
.kpi.verde { border-color: #2D6A4F; }
.kpi.verde .num { color: #2D6A4F; }
.kpi.naranja { border-color: #7D3C08; }
.kpi.naranja .num { color: #7D3C08; }
table { width: 100%; border-collapse: collapse; margin: 10px 0 16px; font-size: 9pt; }
th { background: #1E3A5F; color: white; padding: 7px 10px; text-align: left; }
td { padding: 6px 10px; border-bottom: 1px solid #E0E0E0; }
tr:nth-child(even) td { background: #F5F8FB; }
tr.vital td { background: #E9F7EF; font-weight: bold; }
tr.alerta td { background: #FDEDEC; }
.badge { display: inline-block; padding: 2px 8px; border-radius: 10px;
         font-size: 8pt; font-weight: bold; }
.badge.verde { background: #E9F7EF; color: #1A6B3C; }
.badge.rojo  { background: #FDEDEC; color: #922B21; }
.badge.gris  { background: #F0F0F0; color: #666; }
.bloque { border: 1px solid #D0DCE8; border-radius: 6px;
          padding: 14px 18px; margin: 14px 0; background: #FAFCFF; }
.causa-row { display: flex; gap: 0; margin: 8px 0; }
.causa-nivel { background: #1E3A5F; color: white; padding: 8px 12px;
               font-size: 8pt; font-weight: bold; min-width: 130px;
               border-radius: 4px 0 0 4px; display: flex; align-items: center; }
.causa-texto { background: #F0F5FA; padding: 8px 14px; font-size: 9pt;
               flex: 1; border-radius: 0 4px 4px 0;
               border: 1px solid #D0DCE8; border-left: none; }
.causa-row.alerta .causa-nivel { background: #922B21; }
.causa-row.alerta .causa-texto { background: #FDEDEC; }
.causa-row.ok .causa-nivel { background: #2D6A4F; }
.causa-row.ok .causa-texto { background: #E9F7EF; }
.nota { background: #FFF8E8; border-left: 4px solid #7D3C08;
        padding: 10px 14px; border-radius: 4px;
        font-size: 8.5pt; color: #555; margin-top: 20px; }
.pie { font-size: 7.5pt; color: #888; text-align: center;
       margin-top: 28px; padding-top: 10px; border-top: 1px solid #E0E0E0; }
.img-wrap { margin: 16px 0; }
.img-wrap img { border: 1px solid #E0E0E0; }
"""

# ─────────────────────────────────────────────
# FILAS TABLAS
# ─────────────────────────────────────────────
filas_acc = ''
acciones = [
    ('Bono diferencial Turno ' + str(turno_max_rot),
     'Compensar carga logistica del turno con mayor rotacion', '1-2 meses', 'Alto'),
    ('Revision salarial cada 180 dias',
     'Alerta para empleados sin ajuste mayor a 180 dias', '1 mes', 'Alto'),
    ('Politica permisos flexible Turno ' + str(turno_max_rot),
     'Convertir faltas no avisadas en permisos formales', '2 semanas', 'Medio-Alto'),
    ('Seguimiento semana 4 y 8 — nuevos ingresos',
     'Reducir rotacion temprana (' + str(round(len(df[df['rotacion_temprana']==1])/len(df)*100)) + '% del total)',
     '1 mes', 'Medio'),
]
for accion, desc, plazo, impacto in acciones:
    badge = 'rojo' if 'Alto' in impacto else 'verde'
    filas_acc += (
        '<tr><td><strong>' + accion + '</strong></td><td>' + desc + '</td>'
        '<td style="text-align:center">' + plazo + '</td>'
        '<td style="text-align:center"><span class="badge ' + badge + '">' + impacto + '</span></td></tr>'
    )

filas_ranking = ''
for i, row in df_ranking.head(20).iterrows():
    acum  = row['Acumulado_%']
    clase = 'vital' if acum <= 80 else ''
    badge = '<span class="badge verde">VITAL</span>' if acum <= 80 else '<span class="badge gris">Trivial</span>'
    bar_w = int(row['Importancia_%'] / df_ranking.iloc[0]['Importancia_%'] * 100)
    op    = '0.9' if acum <= 80 else '0.25'
    barra = '<div style="background:#1E3A5F;height:8px;width:' + str(bar_w) + '%;border-radius:4px;opacity:' + op + '"></div>'
    filas_ranking += (
        '<tr class="' + clase + '"><td style="text-align:center">' + str(i) + '</td>'
        '<td>' + str(row['Variable']) + '</td>'
        '<td style="text-align:center">' + str(row['Importancia_%']) + '%</td>'
        '<td style="text-align:center">' + str(acum) + '%</td>'
        '<td style="min-width:100px">' + barra + '</td>'
        '<td style="text-align:center">' + badge + '</td></tr>'
    )

filas_turno = ''
for turno, rd in turno_stats.iterrows():
    clase = 'alerta' if turno == turno_max_rot else ''
    filas_turno += (
        '<tr class="' + clase + '"><td><strong>Turno ' + str(turno) + '</strong></td>'
        '<td style="text-align:center">' + str(int(rd['N'])) + '</td>'
        '<td style="text-align:center">' + str(rd['avg_faltas']) + '</td>'
        '<td style="text-align:center">' + str(round(rd['avg_dias'])) + '</td>'
        '<td style="text-align:center">$' + str(rd['avg_salario']) + '</td>'
        '<td style="text-align:center">' + str(rd['avg_permisos']) + '</td></tr>'
    )

filas_pareto = ''
for _, row in mot_data.iterrows():
    acum  = row['acum']
    clase = 'vital' if acum <= 80 else ''
    badge = '<span class="badge verde">VITAL</span>' if acum <= 80 else '<span class="badge gris">Trivial</span>'
    filas_pareto += (
        '<tr class="' + clase + '"><td>' + str(row['Cat']) + '</td>'
        '<td style="text-align:center">' + str(int(row['N'])) + '</td>'
        '<td style="text-align:center">' + str(row['pct']) + '%</td>'
        '<td style="text-align:center">' + str(acum) + '%</td>'
        '<td style="text-align:center">' + badge + '</td></tr>'
    )

# hipotesis badges
h2_badge = 'CONFIRMADA' if len(df[df['rotacion_temprana']==1]) > 1 else 'PENDIENTE — pocos datos'
h2_color = 'verde' if len(df[df['rotacion_temprana']==1]) > 1 else 'gris'

# ─────────────────────────────────────────────
# INFORME EJECUTIVO
# ─────────────────────────────────────────────
html_ejecutivo = (
'<!DOCTYPE html><html lang="es"><head><meta charset="UTF-8">'
'<title>Informe Ejecutivo Rotacion</title>'
'<style>' + CSS + '</style></head><body><div class="page">'

'<div class="header-box">'
'<h1>Analisis de Rotacion de Personal</h1>'
'<div class="sub">Informe Ejecutivo &nbsp;|&nbsp; ' + hoy +
' &nbsp;|&nbsp; Registros analizados: ' + str(len(df)) + '</div>'
'</div>'

'<h2>Hallazgos Clave</h2>'
'<div class="kpi-row">'
'<div class="kpi rojo"><div class="num">' + str(turno_max_pct) + '%</div>'
'<div class="lbl">bajas en Turno ' + str(turno_max_rot) + '</div></div>'
'<div class="kpi rojo"><div class="num">' + str(round(motivo_top1_pct)) + '%</div>'
'<div class="lbl">son "' + motivo_top1 + '"</div></div>'
'<div class="kpi naranja"><div class="num">' + str(round(t_max['avg_dias'])) + 'd</div>'
'<div class="lbl">sin aumento — Turno ' + str(turno_max_rot) + '</div></div>'
'<div class="kpi verde"><div class="num">' + str(n_vital) + '</div>'
'<div class="lbl">variables explican el 80% de la rotacion</div></div>'
'</div>'

'<h2>Causa Raiz Identificada</h2>'
'<div class="bloque">'
'<div class="causa-row alerta"><div class="causa-nivel">SINTOMA</div>'
'<div class="causa-texto">Alta rotacion concentrada en Turno ' + str(turno_max_rot) +
' (' + str(turno_max_pct) + '% de las bajas)</div></div>'

'<div class="causa-row"><div class="causa-nivel">POR QUE 1</div>'
'<div class="causa-texto">' + str(round(motivo_top1_pct)) + '% son Baja por Faltas — '
'desconexion progresiva, no renuncia activa</div></div>'

'<div class="causa-row"><div class="causa-nivel">POR QUE 2</div>'
'<div class="causa-texto">Turno ' + str(turno_max_rot) + ' promedia ' +
str(round(float(t_max['avg_faltas']), 1)) + ' faltas y ' +
str(round(float(t_max['avg_permisos']), 1)) + ' permisos — no avisa, solo falta</div></div>'

'<div class="causa-row"><div class="causa-nivel">POR QUE 3</div>'
'<div class="causa-texto">Empleados del Turno ' + str(turno_max_rot) +
' mencionan en encuesta: distancia, transporte y condiciones del turno</div></div>'

'<div class="causa-row alerta"><div class="causa-nivel">POR QUE 4</div>'
'<div class="causa-texto">Turno ' + str(turno_max_rot) + ': ' +
str(round(float(t_max['avg_dias']))) + ' dias sin aumento vs ' +
str(round(float(t_min['avg_dias']))) + ' del Turno ' + str(turno_min_rot) +
' — sin diferencial salarial</div></div>'

'<div class="causa-row ok"><div class="causa-nivel">CAUSA RAIZ</div>'
'<div class="causa-texto"><strong>Turno ' + str(turno_max_rot) +
' sin compensacion diferenciada + ' + str(round(float(t_max['avg_dias']))) +
' dias sin aumento + politica de permisos rigida = ausentismo progresivo -> baja por faltas'
'</strong></div></div>'
'</div>'

'<h2>Pareto 80/20</h2>'
'<div class="img-wrap">' + img_tag(b64_pareto) + '</div>'

'<h2>Plan de Accion</h2>'
'<table><tr><th>Accion</th><th>Descripcion</th>'
'<th style="width:80px">Plazo</th><th style="width:80px">Impacto</th></tr>'
+ filas_acc +
'</table>'

'<div class="nota"><strong>Nota metodologica:</strong> Basado en ' + str(len(df)) +
' registros de bajas. Analisis descriptivo — no prueba causalidad. '
'Se recomienda validar con el equipo operativo. Con 50+ registros los resultados '
'seran estadisticamente mas solidos.</div>'

'<div class="pie">Pipeline de Analisis de Rotacion v1.0 &nbsp;|&nbsp;'
' Python · scikit-learn · pandas · matplotlib &nbsp;|&nbsp; ' + hoy + '</div>'
'</div></body></html>'
)

# ─────────────────────────────────────────────
# INFORME TECNICO
# ─────────────────────────────────────────────
html_tecnico = (
'<!DOCTYPE html><html lang="es"><head><meta charset="UTF-8">'
'<title>Informe Tecnico Rotacion</title>'
'<style>' + CSS + '</style></head><body><div class="page">'

'<div class="header-box">'
'<h1>Analisis de Rotacion de Personal</h1>'
'<div class="sub">Informe Tecnico Completo &nbsp;|&nbsp; ' + hoy +
' &nbsp;|&nbsp; Registros: ' + str(len(df)) +
' &nbsp;|&nbsp; Variables procesadas: ' + str(len(feature_names)) + '</div>'
'</div>'

'<h2>1. Hipotesis del Analisis</h2>'
'<div class="bloque">'

'<h3>H1 — Concentracion</h3>'
'<p>La rotacion no esta distribuida uniformemente. Un subconjunto pequeno de variables, '
'motivos y unidades organizacionales explica la mayoria de las bajas.</p>'
'<p><strong>Resultado:</strong> <span class="badge verde">CONFIRMADA</span> &nbsp; '
+ str(n_vital) + ' de ' + str(len(feature_names)) + ' variables explican el 80%. '
'Motivo "' + motivo_top1 + '" = ' + str(round(motivo_top1_pct)) + '%. '
'Turno ' + str(turno_max_rot) + ' = ' + str(turno_max_pct) + '% de las bajas.</p>'

'<h3>H2 — Perfil diferenciado por antiguedad</h3>'
'<p>Los empleados que rotan en los primeros 3 meses tienen un perfil distinto.</p>'
'<p><strong>Resultado:</strong> <span class="badge ' + h2_color + '">' + h2_badge + '</span> &nbsp; '
'Rotacion temprana: ' + str(len(df[df['rotacion_temprana']==1])) + ' casos '
'(' + str(round(len(df[df['rotacion_temprana']==1])/len(df)*100)) + '% del total).</p>'

'<h3>H3 — Desconexion progresiva</h3>'
'<p>La baja por faltas no es una decision activa sino el resultado de condiciones no atendidas.</p>'
'<p><strong>Resultado:</strong> <span class="badge verde">CONFIRMADA</span> &nbsp; '
'Turno ' + str(turno_max_rot) + ' tiene ' + str(round(float(t_max['avg_permisos']), 1)) +
' permisos promedio vs ' + str(round(float(t_min['avg_permisos']), 1)) +
' del Turno ' + str(turno_min_rot) + ' y ' + str(round(float(t_max['avg_dias']))) +
' dias sin aumento.</p>'

'<h3>H4 — Variable dominante accionable</h3>'
'<p>Existe una combinacion de 2-3 variables que describe el 80% de la rotacion y son accionables desde RH.</p>'
'<p><strong>Resultado:</strong> <span class="badge verde">CONFIRMADA</span> &nbsp; '
'Variables: ' + var1 + ', ' + var2 + ' y ' + var3 + ' encabezan el ranking del PCA.</p>'
'</div>'

'<h2>2. Analisis de Componentes Principales (PCA)</h2>'
'<p>Se procesaron <strong>' + str(len(feature_names)) + ' variables</strong> y se requirieron '
'<strong>' + str(n_comp) + ' componentes</strong> para explicar el 80% de la varianza.</p>'
'<div class="img-wrap">' + img_tag(b64_pca) + '</div>'

'<h3>Varianza por componente</h3>'
'<table><tr><th>Componente</th><th>Varianza explicada</th>'
'<th>Varianza acumulada</th><th>Seleccionado</th></tr>'
+ ''.join([
    '<tr class="' + ('vital' if round(float(np.cumsum(var_ratio)[i])*100, 1) <= 80 else '') + '">'
    '<td>CP' + str(i+1) + '</td>'
    '<td style="text-align:center">' + str(round(float(v)*100, 1)) + '%</td>'
    '<td style="text-align:center">' + str(round(float(np.cumsum(var_ratio)[i])*100, 1)) + '%</td>'
    '<td style="text-align:center">' + ('Si' if i < n_comp else 'No') + '</td></tr>'
    for i, v in enumerate(var_ratio)
]) +
'</table>'

'<h3>Ranking de variables por importancia</h3>'
'<table><tr><th>#</th><th>Variable</th><th>Importancia</th>'
'<th>Acumulado</th><th>Peso visual</th><th>Zona</th></tr>'
+ filas_ranking + '</table>'

'<h3>Heatmap de Loadings</h3>'
'<div class="img-wrap">' + img_tag(b64_loadings) + '</div>'

'<h2>3. Analisis Estadistico por Turno</h2>'
'<div class="img-wrap">' + img_tag(b64_turno) + '</div>'
'<table><tr><th>Turno</th><th>N Bajas</th><th>Prom. Faltas</th>'
'<th>Dias sin Aumento</th><th>Salario</th><th>Permisos</th></tr>'
+ filas_turno + '</table>'

'<h2>4. Pareto 80/20</h2>'
'<div class="img-wrap">' + img_tag(b64_pareto) + '</div>'
'<h3>Dimension 1 — Motivo de Salida</h3>'
'<table><tr><th>Motivo</th><th>N</th><th>%</th><th>Acumulado</th><th>Zona</th></tr>'
+ filas_pareto + '</table>'

'<h2>5. Causa Raiz — 5 Porques con Datos</h2>'
'<div class="bloque">'
'<div class="causa-row alerta"><div class="causa-nivel">SINTOMA</div>'
'<div class="causa-texto">Turno ' + str(turno_max_rot) + ' concentra el ' +
str(turno_max_pct) + '% de las bajas <em>(Fuente: Pareto Dim. 3)</em></div></div>'

'<div class="causa-row"><div class="causa-nivel">POR QUE 1</div>'
'<div class="causa-texto">' + str(round(motivo_top1_pct)) + '% son Baja por Faltas — '
'desconexion progresiva <em>(Fuente: Pareto Dim. 1)</em></div></div>'

'<div class="causa-row"><div class="causa-nivel">POR QUE 2</div>'
'<div class="causa-texto">Turno ' + str(turno_max_rot) + ': ' +
str(round(float(t_max['avg_faltas']), 1)) + ' faltas, ' +
str(round(float(t_max['avg_permisos']), 1)) + ' permisos '
'<em>(Fuente: Cruce Turno x PCA)</em></div></div>'

'<div class="causa-row"><div class="causa-nivel">POR QUE 3</div>'
'<div class="causa-texto">Encuesta: distancia, transporte, condiciones del turno '
'<em>(Fuente: Encuesta x Turno)</em></div></div>'

'<div class="causa-row alerta"><div class="causa-nivel">POR QUE 4</div>'
'<div class="causa-texto">Turno ' + str(turno_max_rot) + ': ' +
str(round(float(t_max['avg_dias']))) + ' dias sin aumento vs ' +
str(round(float(t_min['avg_dias']))) + ' del Turno ' + str(turno_min_rot) +
' — sin diferencial salarial <em>(Fuente: PCA CP1 + Estadistico)</em></div></div>'

'<div class="causa-row ok"><div class="causa-nivel">CAUSA RAIZ</div>'
'<div class="causa-texto"><strong>Turno ' + str(turno_max_rot) +
' sin compensacion diferenciada + ' + str(round(float(t_max['avg_dias']))) +
' dias sin aumento + politica de permisos rigida = ausentismo progresivo -> baja por faltas'
'</strong> <em>(PCA + Pareto + Encuesta)</em></div></div>'
'</div>'

'<div class="nota"><strong>Limitaciones:</strong> Basado en ' + str(len(df)) +
' registros sin grupo de control. Analisis descriptivo y exploratorio. '
'Se recomienda 50+ registros para resultados estadisticamente confiables.</div>'

'<div class="pie">Informe Tecnico — Pipeline de Analisis de Rotacion v1.0 &nbsp;|&nbsp;'
' Python · scikit-learn · pandas · matplotlib · seaborn &nbsp;|&nbsp; ' + hoy + '</div>'
'</div></body></html>'
)

# ─────────────────────────────────────────────
# DESCARGAR
# ─────────────────────────────────────────────
nombre_ej  = 'Informe_Ejecutivo_Rotacion_' + hoy_archivo + '.html'
nombre_tec = 'Informe_Tecnico_Rotacion_'   + hoy_archivo + '.html'

with open(nombre_ej,  'w', encoding='utf-8') as f:
    f.write(html_ejecutivo)
with open(nombre_tec, 'w', encoding='utf-8') as f:
    f.write(html_tecnico)

files.download(nombre_ej)
files.download(nombre_tec)

print('Descargados:')
print('  ' + nombre_ej  + '  <- Ejecutivo (para direccion)')
print('  ' + nombre_tec + '  <- Tecnico completo (con graficas y tablas)')
print()
print('Para PDF: abre en Chrome -> Ctrl+P -> Guardar como PDF')

---
## ✅ Análisis completado

### Archivos generados
- **`Analisis_Rotacion_Completo.xlsx`** — descargado automáticamente
- Gráficas mostradas en el notebook (puedes guardarlas con clic derecho)

### Para actualizar con datos nuevos
1. Ve a `Runtime > Run all` (`Ctrl+F9`)
2. Sube el nuevo archivo Excel cuando te lo pida la Sección 1
3. Todo se recalcula automáticamente

### Próximo paso recomendado
**Paso 3 — Modelo Predictivo:** con las variables identificadas aquí, construir un modelo que prediga qué empleados activos tienen mayor riesgo de rotar.
